# Day 44 — Model deployment basics: save model & FastAPI
Objectives:
- Save/load models (joblib).
- Build a minimal FastAPI endpoint.
- Send a sample request.
Note: Running the server requires a terminal (see code comments).


In [ ]:
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

artifact_dir = Path('artifacts/day44')
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / 'model.joblib'
X,y = load_iris(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
print('test acc:', clf.score(Xte,yte))
joblib.dump(clf, model_path)


## Minimal FastAPI app (app.py)
Create a file `app.py` in this folder:
```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()
model = joblib.load(model_path)

class IrisFeatures(BaseModel):
    features: list

@app.post('/predict')
def predict(data: IrisFeatures):
    X = np.array([data.features])
    pred = model.predict(X).tolist()[0]
    return {'prediction': int(pred)}
```
Run server:
```bash
uvicorn app:app --reload
```
Send a request (new terminal):
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```


## Learner exercises and progressive hints

1. Add input validation and friendly error behavior.
2. Return the class name as well as the numeric class identifier.
3. Create a minimal runtime dependency file for this API.

### Progressive hints

1. Let Pydantic reject the wrong length or nonnumeric values. Do not catch every
   exception and turn programming defects into vague client errors.
2. Keep the mapping beside the model metadata and test all valid numeric class
   identifiers.
3. Include only direct runtime imports. Pin or lock versions through the
   repository tooling rather than copying the entire development environment.

### Additional mastery practice

Make an API boundary explicit: validate shape and meaning, map model outputs to a versioned schema, and test behavior without relying on a manually running server.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Boundary-case testing:** Write API tests for a missing feature, an extra feature, a string, NaN/infinity, wrong feature count, and one valid request. State the expected status-code family for each.
   **Progressive hint:** Use FastAPI TestClient so validation can be tested in-process. Malformed client input is 4xx; unexpected service failure is 5xx.
5. **Batch contract:** Design a `/predict-batch` request and response with stable row IDs, a maximum batch size, ordered results, and per-request model metadata.
   **Progressive hint:** Validate the entire batch before scoring or define explicit partial failure semantics. Never rely only on list position to identify rows.
6. **Artifact-compatibility check:** At startup, validate model version, expected feature schema, and class metadata before accepting traffic. Explain why loading a pickle from an untrusted source is unsafe.
   **Progressive hint:** Persist a small manifest beside the artifact and compare required fields. Python pickle/joblib loading can execute code.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Boundary-case testing


# Practice 5 — Batch contract


# Practice 6 — Artifact-compatibility check
